In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("/sujin/PycharmProjects/rllm")

import json

from pathlib import Path
from rllm.data import DatasetRegistry, Dataset
from rllm.eval.agent_loader import load_agent, register_agent, list_agents
from rllm.runner import Runner
from rllm.types import AgentConfig, Task
from rllm.eval.evaluator_loader import load_evaluator, list_evaluators, register_evaluator
from tqdm import tqdm
from utils.mpr import MultipleProcessRunnerSimplifier

14:40:20 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: _ssl.c:999: The handshake operation timed out. Falling back to local backup.


# Tokenizer check

In [20]:
from transformers import AutoTokenizer

_CALCULATOR_DESCRIPTION = (
    "Evaluate a mathematical expression. Supports operators +, -, *, /, //, ** (power), "
    "% (modulo), and parentheses. Functions: abs, round, min, max, sum, pow, sqrt, exp, "
    "log, log2, log10, sin, cos, tan, asin, acos, atan, atan2, sinh, cosh, tanh, floor, "
    "ceil, trunc, factorial, gcd, lcm, comb (aka binom), perm, degrees, radians. "
    "Constants: pi, e, tau."
)

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": _CALCULATOR_DESCRIPTION,
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The expression to evaluate, e.g. 'sqrt(64 + 225)' or 'binom(10, 3) * pi'",
                    }
                },
                "required": ["expression"],
            },
        },
    }
]

model_name = "/sujin/Models/Qwen/Qwen3-4B"

SYSTEM_PROMPT = """\
You are a math assistant that solves competition math problems step by step.
You have access to a calculator tool and you MUST use it.

IMPORTANT rules you must follow:
1. You MUST call the calculator tool at least once before giving your final answer. \
Answers given without any prior tool call will be marked wrong.
2. Do NOT perform arithmetic in your head — every computation must go through the calculator.
3. Break the problem into small steps. Make one tool call per step, then reason about the result.
4. When you have the final answer, put it in \\boxed{ANSWER} in your response.
"""

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
    tools=TOOLS
)

print(len(text))
print(tokenizer.is_fast)

1718
True


In [13]:
def do(process_id, idx, item, writer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
            tools=TOOLS
            )

items = [i for i in range(1000)]
mprs = MultipleProcessRunnerSimplifier(items, do, n_process=256)
mprs.run()

[Errno 25] Inappropriate ioctl for device
Can't get terminal size, set terminal_y = None
7Total: [##########################################] 100% 1000/1000 [00:00:04 < 00:00:00, 222.08it/s]8

Aggregating results...: 256it [00:00, 411080.33it/s]


[]

# Single test

In [117]:
dataset = DatasetRegistry.load_dataset("math500", "test")

item = dataset[102]
print(item.keys())   # → dict_keys(['question', 'answer', ...)
print(item["question"])

for item in dataset:
    if "If a snack-size tin of peaches has $40$" in item["question"]:
        print(item["question"])
        break

dict_keys(['question', 'ground_truth', 'data_source'])
Factor the expression $ab+5b+2a+10$.
If a snack-size tin of peaches has $40$ calories and is $2\%$ of a person's daily caloric requirement, how many calories fulfill a person's daily caloric requirement?


In [68]:
register_agent("math_tool_agent", "cookbooks.math_tool_agent.sync_math_tool_agent:math_tool_agent")
list_agents()

register_evaluator("math_tool_evaluator", "cookbooks.math_tool_agent.evaluator:math_tool_evaluator")
list_evaluators()

[{'name': 'math_tool_evaluator',
  'source': 'registered',
  'type': 'cookbooks.math_tool_agent.evaluator:math_tool_evaluator'},
 {'name': 'bfcl_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.bfcl:evaluate'},
 {'name': 'code_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.code:evaluate'},
 {'name': 'countdown_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.countdown:evaluate'},
 {'name': 'depth_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.depth:evaluate'},
 {'name': 'f1_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.f1:evaluate'},
 {'name': 'ifeval_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.ifeval:evaluate'},
 {'name': 'iou_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.iou:evaluate'},
 {'name': 'llm_equality_reward_fn',
  'source': 'built-in',
  'type': 'rllm.eval.reward_fns.llm_equality:evaluate'},
 {'name': 'llm_judge_reward_fn',
  's

In [104]:
agent = load_agent("math_tool_agent")

task = Task(
    id="math500",
    instruction=item["question"] + "\n用中文来回答",
    # instruction="列出你所有可用的工具以及他们的描述给我",
    metadata=item,
    dataset_dir=Path("/root/.rllm/datasets/math500"),  # 相对路径
)

config = AgentConfig(
        base_url="http://localhost:30000/v1",  # LiteLLM proxy 地址
        model="/sujin/Models/Qwen/Qwen3-4B",
        session_uid="test_session_uid"
    )

episode = agent.run(task, config)

In [105]:
print("number of trajectories:", len(episode.trajectories))
print("number of steps in the first trajectory:", len(episode.trajectories[0].steps))
print(json.dumps(episode.trajectories[0].steps[-1].chat_completions, indent=4, ensure_ascii=False))

# text = tokenizer.apply_chat_template(
#     episode.trajectories[0].steps[-1].chat_completions,
#     tokenize=False,
#     add_generation_prompt=True,
#     enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
#     tools=TOOLS
# )
#
# print(text)

evaluator = load_evaluator("math_tool_evaluator")  # 或其他 registered evaluator
result = evaluator.evaluate(task.metadata, episode)
print(result)

number of trajectories: 1
number of steps in the first trajectory: 2
[
    {
        "role": "system",
        "content": "You are a math assistant that solves competition math problems step by step.\nYou have access to a calculator tool and you MUST use it.\n\nIMPORTANT rules you must follow:\n1. You MUST call the calculator tool at least once before giving your final answer. Answers given without any prior tool call will be marked wrong.\n2. Do NOT perform arithmetic in your head — every computation must go through the calculator.\n3. Break the problem into small steps. Make one tool call per step, then reason about the result.\n4. When you have the final answer, put it in \\boxed{ANSWER} in your response.\n"
    },
    {
        "role": "user",
        "content": "What is the least positive integer multiple of 30 that can be written with only the digits 0 and 2?\n用中文来回答"
    },
    {
        "role": "assistant",
        "content": "<think>\n好的，我现在要解决的问题是：找出最小的正整数倍数30，且这个数只能由0和2组成。首先

In [3]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:30000/v1", api_key="EMPTY")
# client = OpenAI(base_url="http://172.98.0.90:37863/v1", api_key="EMPTY")

messages: list[dict] = [
    {"role": "system", "content": ""},
    {"role": "user", "content": "你好"},
]

resp = client.chat.completions.create(
    model="/sujin/Models/Qwen/Qwen3-4B",
    messages=messages,
    temperature=0.6,
    max_tokens=1000,
    max_completion_tokens=1000,
    timeout=100,
)
content = resp.choices[0].message.content or ""
print(content)

<think>
嗯，用户发来“你好”，这是一条非常简单的问候语。首先，我需要回应这个问候，保持友好和专业的态度。用户可能只是想开始对话，或者有更具体的问题需要解决。我需要确定他们需要什么帮助，可能是在询问某个问题，或者只是需要一个简单的回应。

接下来，我应该考虑用户的潜在需求。他们可能对某个话题感兴趣，或者遇到了问题需要帮助。例如，他们可能在询问技术问题、生活建议，或者只是想进行闲聊。作为AI助手，我需要保持开放和包容的态度，鼓励用户进一步说明他们的需求。

同时，我需要确保回应简洁明了，避免使用过于复杂的语言。用户可能希望得到快速而直接的回答，所以需要在回复中提供足够的信息，同时保持友好。另外，我需要检查是否有任何可能的误解，比如用户可能用“你好”作为测试，或者有其他隐藏的意图。

最后，我应该保持积极的态度，让用户感到被重视和支持。通过友好的问候和开放的提问，可以促进更深入的对话，帮助用户更好地解决问题或获得所需的信息。
</think>

你好！很高兴见到你。我是Qwen，一个由通义实验室研发的大型语言模型。我能够帮助你回答问题、创作内容、提供建议等。无论你有什么需求，都可以告诉我，我会尽力协助你！😊 你今天想聊什么？


# Complete Test

In [ ]:
register_evaluator("math_tool_evaluator", "cookbooks.math_tool_agent.evaluator:math_tool_evaluator")

dataset_id = "math500"
dataset = DatasetRegistry.load_dataset(dataset_id, "test")
agent = load_agent("math_tool_agent")
evaluator = load_evaluator("math_tool_evaluator")

def do(process_id, idx, item, writer):
    task = Task(
        id=dataset_id,
        instruction=item["question"],
        metadata=item,
        dataset_dir=Path(f"/root/.rllm/datasets/{dataset_id}"),  # 相对路径
    )

    config = AgentConfig(
        base_url="http://localhost:30000/v1",  # LiteLLM proxy 地址
        model="/sujin/Models/Qwen/Qwen3-4B",
        session_uid="notebook-0",
    )

    episode = agent.run(task, config)
    result = evaluator.evaluate(task.metadata, episode)

    # content = episode.trajectories[0].steps[0].action
    # print(len(content))

    # from transformers import AutoTokenizer
    # tokenizer = AutoTokenizer.from_pretrained("/sujin/Models/Qwen/Qwen3-4B")
    # tokens = tokenizer.encode(content)
    # print(len(tokens))

    writer.write(f"{int(result.is_correct)}\n")


items = [item for item in dataset]
mprs = MultipleProcessRunnerSimplifier(items, do, n_process=256, return_results=True)
outputs = mprs.run()

In [26]:
print(len(outputs))
results = [int(o) for o in outputs]
acc = sum(results) / len(results)
print(acc)

0


ZeroDivisionError: division by zero